# 08 — Two accepted non-LLM improvements

Review iterations 5–7, including the rejected intermediate query policy. No new experiment starts when this notebook runs. All comparisons use 140 answerable train and 23 frozen unanswerable train questions; no test performance has been measured.

**Accepted improvement 1:** infer the source contract from the full query, then rank passages in that source using only the clause question.

**Accepted improvement 2:** remove generic wording from the clause question; if focused terms have no positive-score matches, use the accepted unfiltered clause query in the same inferred contract. Both stages use BM25 and original document offsets, without gold IDs at inference.

The second improvement addresses the rejected iteration 6's false abstentions. Fallback helps retain answers but is not evidence that an answer exists.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from improve import read_json, read_log
from retrieval import BM25Retriever, load_inputs

runs = {r['iteration']:r for r in read_log(ROOT / 'experiments/loop_log.jsonl')}
rows = []
for label, metrics, accepted in [
    ('baseline', runs[7]['metrics']['baseline'], None),
    ('5: document then clause', runs[5]['metrics']['candidate'], runs[5]['decision']['accepted']),
    ('6: focused terms without fallback', runs[6]['metrics']['candidate'], runs[6]['decision']['accepted']),
    ('7: focused terms with fallback', runs[7]['metrics']['candidate'], runs[7]['decision']['accepted'])]:
    at5 = next(m for m in metrics['retrieval'] if m['k'] == 5)
    rows.append(dict(approach=label, accepted=accepted, precision_at_5=at5['precision'],
                     recall_at_5=at5['recall'], benchmark_precision=at5['benchmark_precision'],
                     benchmark_recall=at5['benchmark_recall'],
                     false_abstention=metrics['false_abstention'], correct_abstention=metrics['correct_abstention']))
comparison = pd.DataFrame(rows)
display(comparison)
print('Accepted improvements:', sum(r['decision']['accepted'] for r in runs.values()))
print('Completed distinct iterations:', max(runs))

,approach,accepted,precision_at_5,recall_at_5,benchmark_precision,benchmark_recall,false_abstention,correct_abstention
0,baseline,None,0.017433,0.181647,0.017594,0.192427,0.000000,0.000000
1,5: document then clause,True,0.051525,0.471210,0.053680,0.521066,0.000000,0.000000
2,6: focused terms without fallback,False,0.058116,0.507164,0.060665,0.562691,0.085714,0.043478
3,7: focused terms with fallback,True,0.062782,0.535021,0.065070,0.591568,0.000000,0.000000


Accepted improvements: 2
Completed distinct iterations: 7


## Why the intermediate trial was rejected

Iteration 6 increased recall@5 to 50.72% but abstained on 12/140 answerable questions (8.57%), exceeding the 2-point guardrail. It correctly abstained on 1/23 negatives.

Iteration 7 is a new policy, measured against the retained iteration-5 reference. It increases recall by 6.38 percentage points while preserving precision and both reference abstention rates. The 1/23 negative abstention achieved by the rejected intermediate model is lost with fallback, so the final system still needs a better answerability mechanism.

All numerical acceptance rules remain unchanged. No parameter was tuned on test.

In [2]:
for iteration in [5,6,7]:
    print('ITERATION', iteration, runs[iteration]['proposal']['id'])
    print('Before:', runs[iteration]['diagnosis_before'])
    print('After:', runs[iteration]['diagnosis_after'])
    display(pd.Series(runs[iteration]['decision']['checks'], name='Passed'))

ITERATION 5 document_then_clause
Before: {'complete_coverage': 15, 'right_document_wrong_section': 103, 'partial_coverage': 22, 'false_returns': 23}
After: {'complete_coverage': 51, 'right_document_wrong_section': 62, 'partial_coverage': 27}


recall_gain                True
precision                  True
answerable_abstention      True
unanswerable_abstention    True
latency                    True
Name: Passed, dtype: bool

ITERATION 6 clause_query_cleanup
Before: {'complete_coverage': 51, 'right_document_wrong_section': 62, 'partial_coverage': 27, 'false_returns': 23}
After: {'complete_coverage': 55, 'right_document_wrong_section': 44, 'partial_coverage': 29, 'incorrect_abstention': 12}


recall_gain                 True
precision                   True
answerable_abstention      False
unanswerable_abstention     True
latency                     True
Name: Passed, dtype: bool

ITERATION 7 focused_clause_with_fallback
Before: {'complete_coverage': 51, 'right_document_wrong_section': 62, 'partial_coverage': 27, 'false_returns': 23}
After: {'complete_coverage': 57, 'right_document_wrong_section': 50, 'partial_coverage': 33}


recall_gain                True
precision                  True
answerable_abstention      True
unanswerable_abstention    True
latency                    True
Name: Passed, dtype: bool

## Try all approaches

Edit `question` or choose `baseline`, `document_then_clause`, `focused_rejected`, or `focused_fallback`. Each model loads a separate preserved configuration. Questions in the benchmark's `Consider ...; ...` format activate routing. Plain free-form questions use the original full-query search, so these gains must not be generalized to all user phrasing.

In [3]:
approach = 'focused_fallback'
config_files = {
    'baseline':'bm25_baseline',
    'document_then_clause':'bm25_document_then_clause',
    'focused_rejected':'bm25_focused_clause',
    'focused_fallback':'bm25_focused_clause_fallback',
}
_, documents, _ = load_inputs(ROOT)
config = read_json(ROOT / 'configs' / f'{config_files[approach]}.json')
model = BM25Retriever(documents, config)
question = 'Consider the Wireless Content License Agreement between Fox Mobile Entertainment and Glu Mobile; What is the expiration date of this contract?'
hits = model.search(question, k=5)
if hits:
    display(pd.DataFrame(hits)[['score','file_path','start','end','text']])
else:
    print('No relevant passage found.')

,score,file_path,start,end,text
0,2.069180,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."
1,1.082455,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."


## Inspect a successful change and a remaining failure

Use saved predictions to compare coverage at k=5. A gain in train recall does not establish generalization, legal correctness, or reliable abstention.

In [4]:
latest = ROOT / runs[7]['result_directory']
base = read_json(latest / 'baseline.json')['answerable']
new = read_json(latest / 'candidate.json')['answerable']
a = pd.DataFrame(base['per_query'])
b = pd.DataFrame(new['per_query'])
changes = a[a['k']==5][['query_id','recall']].merge(
    b[b['k']==5][['query_id','recall']], on='query_id', suffixes=('_baseline','_improved'))
changes['gain'] = changes['recall_improved']-changes['recall_baseline']
questions = pd.DataFrame(new['predictions'])[['query_id','query']]
display(changes.nlargest(3,'gain').merge(questions,on='query_id'))
display(changes[changes['recall_improved']==0].head(3).merge(questions,on='query_id'))

,query_id,recall_baseline,recall_improved,gain,query
0,41ece44a0165f292b5f7c0cd8f4402ff9844983a576937...,0.0,1.0,1.0,Consider the Cooperation Agreement between HPI...
1,70069a2599a5b9df39ec8e006a110e01d374cea11133eb...,0.0,1.0,1.0,Consider the Sponsorship Agreement between Jac...
2,3788364d7ef118c8a9d898b74e2ceabf9eb7fa44bbea7c...,0.0,1.0,1.0,Consider the Sponsorship Agreement between Exc...


,query_id,recall_baseline,recall_improved,gain,query
0,b5a0948cd3cf1a4070748a4899cab8bea0dde912b91f0f...,0.0,0.0,0.0,Consider the Service Agreement between Bicycle...
1,13d8e76465cd5d5ff8d880d4fcf36657d918c9f750e2aa...,0.0,0.0,0.0,Consider the Service Agreement between Bicycle...
2,505f5239e0ca460f383a732198b07e7d4dd7f828487ada...,0.0,0.0,0.0,Consider the Service Agreement between Bicycle...


## Explain the result

“Contract names were helping us find the right document but distracting the clause ranking. Separating those two jobs produced the first large gain. We then focused the clause query while preserving a fallback for missing vocabulary. Both improvements passed our predeclared checks; the intermediate version was rejected because it wrongly declined answerable questions.”

At k=5, unique-character recall improves **18.16% → 47.12% → 53.50%**, and precision **1.74% → 5.15% → 6.28%**. Reference-compatible recall is **19.24% → 52.11% → 59.16%**. These are distinct scoring conventions.

Business value: more relevant clause coverage with a fixed five-passage review budget and no external model API. Limitations: still substantial irrelevant text, unresolved definitions/cross-references, dependence on structured questions, possible document-routing errors, and 0/23 correct negative abstentions.

Seven distinct non-LLM trials are now complete, including two accepted improvements. The user allows up to ten iterations. Finish the required web/search CLI and core report before optional LLM or hybrid/reranker extensions. Dense retrieval with FAISS can be evaluated separately; no such improvement is assumed or implemented here.